중복된 거래를 어떻게 처리할지 실험하는 노트북 입니다.

In [260]:
import pandas as pd
import glob
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [261]:

# 사용할 컬럼만 지정 (NO 컬럼 제외)
columns_to_use = [
    '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일',
    '거래금액(만원)', '동', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일',
    '거래유형', '중개사소재지', '등기일자'
]

# 병합할 CSV 파일이 들어있는 폴더 경로
folder_path = "../../data/raw/apt_sale"

# 해당 경로의 모든 .csv 파일 리스트 얻기
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# 병합할 데이터프레임 저장할 리스트
df_list = []

# 각 CSV 파일을 순회하며 읽기
for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            encoding='cp949',
            skiprows=15,               # 메타데이터 줄 건너뛰기
            usecols=columns_to_use     # 필요한 컬럼만 불러오기
        )
        df_list.append(df)
        print(f"읽기 완료: {os.path.basename(file)}")
    except Exception as e:
        print(f"오류 발생: {os.path.basename(file)} — {e}")

# 데이터프레임 병합
df = pd.concat(df_list, ignore_index=True)

# 병합된 결과 출력
print(f"\n 병합 완료: 총 {len(merged_df)}건")

읽기 완료: 아파트(매매)_실거래가_20250620103841.csv
읽기 완료: 아파트(매매)_실거래가_20250620103317.csv
읽기 완료: 아파트(매매)_실거래가_20250620103854.csv
읽기 완료: 아파트(매매)_실거래가_20250620103311.csv
읽기 완료: 아파트(매매)_실거래가_20250620103846.csv
읽기 완료: 아파트(매매)_실거래가_20250620103404.csv
읽기 완료: 아파트(매매)_실거래가_20250620103822.csv
읽기 완료: 아파트(매매)_실거래가_20250620132033.csv
읽기 완료: 아파트(매매)_실거래가_20250620132025.csv
읽기 완료: 아파트(매매)_실거래가_20250620132018.csv
읽기 완료: 아파트(매매)_실거래가_20250620103400.csv
읽기 완료: 아파트(매매)_실거래가_20250620132012.csv
읽기 완료: 아파트(매매)_실거래가_20250620100439.csv

 병합 완료: 총 512361건


## 면적당 단가 계산

In [262]:
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',', '').astype(int)
df['면적당 단가(만원)'] = df['거래금액(만원)'] / df['전용면적(㎡)']

## 아파트 나이 계산

In [263]:
df['계약년도'] = df['계약년월'].astype(str).str[:4].astype(int)
df['아파트 나이'] = df['계약년도'] - df['건축년도']

In [264]:

df['구'] = df['시군구'].str.extract(r'(\S+구)')

df.drop([
    '시군구', '번지', '본번', '부번', '동', '층', '매수자', '매도자','계약년도','구',
    '해제사유발생일', '거래유형', '중개사소재지', '등기일자',
], axis=1, inplace=True)

In [265]:
# 계약연-월-일을 기준으로 시계열 정렬
df['계약일자'] = df['계약년월'].astype(str) + df['계약일'].astype(str).str.zfill(2)
df['계약일자'] = pd.to_datetime(df['계약일자'], format='%Y%m%d')

df = df.sort_values('계약일자').reset_index(drop=True)

In [266]:
df = df[['전용면적(㎡)','건축년도','아파트 나이','단지명','도로명','면적당 단가(만원)','거래금액(만원)','계약일자']]

In [267]:
df.head()

,전용면적(㎡),건축년도,아파트 나이,단지명,도로명,면적당 단가(만원),거래금액(만원),계약일자
0,59.69,1999,21,한숲마을대림,방화대로50길 7,861.115765,51400,2020-07-11
1,84.33,2001,19,홍제삼성래미안,통일로 319,960.512273,81000,2020-07-11
2,60.00,2001,19,홍제삼성래미안,통일로 319,1166.666667,70000,2020-07-11
3,59.93,2000,20,홍제원현대(459-0),통일로34길 43,1168.029368,70000,2020-07-11
4,114.72,2000,20,홍제원현대(459-0),통일로34길 43,862.970711,99000,2020-07-11


# 방법 1: 아파트 나이 기준 대치

In [268]:
def process_group(group):
    if (group['아파트 나이'] <= 10).all():
        row = group.iloc[0].copy()
        row['면적당 단가(만원)'] = group['면적당 단가(만원)'].mean()
        return pd.DataFrame([row])
    else:
        min_age = group['아파트 나이'].min()
        return group[group['아파트 나이'] == min_age].iloc[[0]]

way1 = df.groupby(['도로명','단지명','전용면적(㎡)'], group_keys=False).apply(process_group).reset_index(drop=True)


/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_41015/3593404691.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  way1 = df.groupby(['도로명','단지명','전용면적(㎡)'], group_keys=False).apply(process_group).reset_index(drop=True)


# 방법 2: 시계열 가중치 적용 + 이상치 제거

In [269]:
# 이상치 제거 함수 예시 (IQR 방식 등 사용자 정의 필요)
def remove_price_outliers(group):
    q1 = group['거래금액(만원)'].quantile(0.25)
    q3 = group['거래금액(만원)'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    filtered = group[(group['거래금액(만원)'] >= lower) & (group['거래금액(만원)'] <= upper)]
    return filtered

def calculate_alpha_from_age_count(age, count, N=30):
    """
    아파트 나이(age)와 해당 월 중복 거래 수(count)를 바탕으로
    대표 거래가 산정을 위한 시계열 가중치 α를 계산한다.

    α = min(1, max(0, (1 - age / N) * log2(count + 1)))

    ▣ α 의미:
      - 월평균 거래가(평균값)와 최신 거래가의 가중 평균에서
        평균값에 부여되는 신뢰도 가중치
      - 0 ≤ α ≤ 1 사이 값

    ▣ 설계 목적:
      - 연식이 오래된 아파트일수록 가격 변동성이 크므로,
        최신 거래가(P_latest)에 더 높은 비중을 부여
      - 거래 수가 많을수록 평균값의 신뢰도는 높아지므로 α 증가

    ▣ 기준 수명 N의 역할:
      - 아파트가 노후되기 시작하는 시점을 수치화 (기본값 30년)
      - 한국 도시정비법상 재건축 가능 기준도 30년 → 현실적 기준
        · age = 0 → α 최대 (평균가 신뢰도 최대)
        · age = N → α = 0 (평균가 신뢰도 제거, 최신값만 사용)

    Parameters:
        age (float): 아파트 나이 (년 단위)
        count (int): 해당 월 중복 거래 수
        N (int): 기준 수명 (default: 30년)

    Returns:
        float: 가중치 α (0 ~ 1 범위)
    """
    raw_alpha = (1 - age / N) * np.log2(count + 1)
    alpha = max(0, min(1, raw_alpha))
    return alpha

def representative_price(prices, dates, age, N=30):
    """
    월별 이상치 제거된 거래 가격 리스트와 거래일 리스트,
    아파트 나이를 바탕으로 대표 거래가격을 계산한다.

    대표 거래가 = α * 평균 거래가 + (1 - α) * 최신 거래가

    Parameters:
        prices (list or np.ndarray): 이상치 제거 후의 거래 가격 리스트
        dates (list or np.ndarray): 거래일 리스트 (prices와 길이 동일)
        age (float): 아파트 나이
        N (int): 아파트 기준 수명 (기본 30년)

    Returns:
        float or None: 대표 거래 가격. 거래가 없을 경우 None 반환.
    """
    if len(prices) == 0:
        return None  # 거래 없음 → 대표값 계산 불가

    # 가중치 alpha 계산
    count = len(prices)
    alpha = calculate_alpha_from_age_count(age, count, N)

    # 평균 거래가 (P̄)
    avg_price = np.mean(prices)

    # 최신 거래가 (P_latest) → 가장 나중의 날짜 기준
    latest_index = np.argmax(dates)  # 거래일 기준 최대값 인덱스
    latest_price = prices[latest_index]

    # 대표 거래가 계산
    rep_price = alpha * avg_price + (1 - alpha) * latest_price
    return rep_price


def calculate_alpha_row(group, N=30):
    """
    pandas group (같은 월 내 중복 거래 묶음)을 받아서
    alpha 값을 구하고, 해당 그룹의 첫 row에 붙여 반환.

    Parameters:
        group (pd.DataFrame): 월별 중복 거래 묶음
        N (int): 기준 수명

    Returns:
        pd.DataFrame: alpha가 추가된 대표 row 1개
    """
    age = group['아파트 나이'].iloc[0]  # 해당 그룹의 아파트 나이
    count = len(group)  # 그룹 내 거래 수

    alpha = calculate_alpha_from_age_count(age, count, N)

    # 대표 row는 그룹의 첫 row 기준으로 생성
    row = group.iloc[0].copy()
    row['alpha'] = alpha
    return pd.DataFrame([row])

In [270]:
# 1. 이상치 제거
# df_filtered = df.groupby(['도로명', '단지명', '전용면적(㎡)'], group_keys=False)\
#                 .apply(remove_price_outliers)\
#                 .reset_index(drop=True)

# 2. 가중치 α 계산 및 대표 row 추출
way2 = df.groupby(['도로명', '단지명', '전용면적(㎡)'], group_keys=False)\
                  .apply(calculate_alpha_row)\
                  .reset_index(drop=True)

/var/folders/sq/2pgmkw912zj1zpn9tz6pjdmr0000gn/T/ipykernel_41015/821188375.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calculate_alpha_row)\


In [271]:
way2.drop('거래금액(만원)',axis=1, inplace=True)
way1 = way1.sort_values('계약일자').reset_index(drop=True)
way2 = way2.sort_values('계약일자').reset_index(drop=True)